# NLI Zero-Shot Baseline: mDeBERTa-v3 Paragraph-Level Classification

**目标**：用 NLI 范式做段落级宣传技术检测，作为论文中的零样本基线。

**方法**：
- Premise = 段落文本
- Hypothesis = `This text contains [technique].` (Design A) 或 定义增强版 (Design B)
- entailment 概率 → 阈值判断是否存在该技术

**模型**：`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`

**评估集**：SemEval 2023 dev ∩ CheckThat! 2024 overlap (EN=90篇, PL=49篇, RU=48篇)

**执行顺序**：EN Design A → EN Design B → 比较 → 最优设计跑 PL & RU

In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory of this repository
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# SemEval-2023 Task 3 development data
SEMEVAL_DEV_DIR = "your/path/here"  # contains en/, po/, ru/ subdirectories

# CheckThat! 2024 Task 3 data
CHECKTHAT_DIR = "your/path/here"

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


## Cell 1: 环境 & 路径配置

In [ ]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import torch
from transformers import pipeline
from sklearn.metrics import f1_score, precision_score, recall_score

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ============================================================
# 路径配置
# ============================================================
BASE = Path("your/path/here")
DATA_BASE = Path(BASE)

GOLD_SPAN_PATHS = {
    "en": DATA_BASE / "data_analy/overlap_analysis_results/en_overlap_articles/labels-subtask-3-spans.txt",
    "pl": DATA_BASE / "data_analy/overlap_analysis_results/po_overlap_articles/labels-subtask-3-spans.txt",
    "ru": DATA_BASE / "data_analy/overlap_analysis_results/ru_overlap_articles/labels-subtask-3-spans.txt",
}

TECHNIQUE_FILE = DATA_BASE / "technique_list/techniques_subtask3.txt"

ARTICLE_DIRS = {
    "en": Path("your/path/here"  # SemEval 2023 data directory),
    "pl": Path("your/path/here"  # SemEval 2023 data directory),
    "ru": Path("your/path/here"  # SemEval 2023 data directory),
}

OUTPUT_DIR = BASE / "results/nli_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
THRESHOLD_GRID = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
MIN_PARA_LEN = 30  # 段落最小字符数

print("\n路径检查：")
for lang, p in GOLD_SPAN_PATHS.items():
    print(f"  [{lang}] gold spans: {'OK' if p.exists() else 'NOT FOUND'} -> {p}")
for lang, p in ARTICLE_DIRS.items():
    print(f"  [{lang}] articles:   {'OK' if p.exists() else 'NOT FOUND'} -> {p}")
print(f"  technique file: {'OK' if TECHNIQUE_FILE.exists() else 'NOT FOUND'} -> {TECHNIQUE_FILE}")
print(f"  output dir: {OUTPUT_DIR}")

## Cell 2: 技术列表 & Hypothesis 定义（Design A & B）

In [ ]:
# 从文件加载 23 种技术名称
with open(TECHNIQUE_FILE, "r", encoding="utf-8") as f:
    TECHNIQUES = [line.strip() for line in f if line.strip()]

print(f"共 {len(TECHNIQUES)} 种技术：")
for i, t in enumerate(TECHNIQUES):
    print(f"  {i:2d}: {t}")

# -------- Design A: 简单标签名 --------
def get_hypothesis_A(technique: str) -> str:
    readable = technique.replace("_", " ").replace("-", " ")
    return f"This text contains {readable}."

# -------- Design B: 定义增强 -------- 来自 DATA_BASE / "technique_list/Technical_explanation.txt"
TECHNIQUE_DEFINITIONS = {
    "Appeal_to_Authority": "A weight is given to an argument, an idea or information by simply stating that a particular entity considered as an authority is the source of the information.",
    "Appeal_to_Popularity": "This technique gives weight to an argument or idea by justifying it on the basis that allegedly 'everybody' (or the vast majority) agrees with it or 'nobody' disagrees with it.",
    "Appeal_to_Values": "This technique gives weight to an idea by linking it to values seen by the target audience as positive.",
    "Appeal_to_Fear-Prejudice": "This technique aims at promoting or rejecting an idea through the repulsion or fear of the audience towards this idea (e.g., via exploiting some preconceived judgements) or towards its alternative.",
    "Flag_Waving": "Justifying or promoting an idea by exhaling the pride of a group or highlighting the benefits for that specific group.",
    "Causal_Oversimplification": "Assuming a single cause or reason when there are actually multiple causes for an issue.",
    "False_Dilemma-No_Choice": "Sometimes called the either-or fallacy, a false dilemma is a logical fallacy that presents only two options or sides when there actually are many.",
    "Consequential_Oversimplification": "An argument or an idea is rejected and instead of discussing whether it makes sense and/or is valid, the argument affirms, without proof, that accepting the proposition would imply accepting other propositions that are considered negative.",
    "Straw_Man": "This technique consists in making an impression of refuting the argument of the opponent's proposition, whereas the real subject of the argument was not addressed or refuted, but instead replaced with a false one.",
    "Red_Herring": "This technique consists in diverting the attention of the audience from the main topic being discussed, by introducing another topic.",
    "Whataboutism": "A technique that attempts to discredit an opponent's position by charging them with hypocrisy without directly disproving their argument.",
    "Slogans": "A brief and striking phrase that may include labeling and stereotyping. Slogans tend to act as emotional appeals.",
    "Appeal_to_Time": "The argument is centered around the idea that time has come for a particular action. The very timeliness of the idea is part of the argument.",
    "Conversation_Killer": "This includes words or phrases that discourage critical thought and meaningful discussion about a given topic.",
    "Loaded_Language": "Use of specific words and phrases with strong emotional implications (either positive or negative) to influence and to convince the audience that an argument is valid.",
    "Repetition": "The speaker uses the same word, phrase, story, or imagery repeatedly with the hope that the repetition will lead to persuade the audience.",
    "Exaggeration-Minimisation": "This technique consists of either representing something in an excessive manner – by making things larger, better, worse – or by making something seem less important or smaller than it really is, downplaying the statements and ignoring the arguments and the accusations made by an opponent.",
    "Obfuscation-Vagueness-Confusion": "This fallacy uses words that are deliberately not clear, so that the audience may have its own interpretations.",
    "Name_Calling-Labeling": "A form of argument in which loaded labels are directed at an individual or a group, typically in an insulting or demeaning way.",
    "Doubt": "Casting doubt on the character or the personal attributes of someone or something in order to question their general credibility or quality, instead of using a proper argument related to the topic.",
    "Guilt_by_Association": "Attacking the opponent or an activity by associating it with another group, activity, or concept that has sharp negative connotations for the target audience.",
    "Appeal_to_Hypocrisy": "The target of the technique is attacked on its reputation by charging them with hypocrisy or inconsistency.",
    "Questioning_the_Reputation": "This technique is used to attack the reputation of the target by making strong negative claims about it, focusing specially on undermining its character and moral stature rather than relying on an argument about the topic.",
}

def get_hypothesis_B(technique: str) -> str:
    definition = TECHNIQUE_DEFINITIONS.get(technique, "")
    readable = technique.replace("_", " ").replace("-", " ")
    if definition:
        return f"This text uses the propaganda technique of {readable}: {definition}"
    else:
        return f"This text uses the propaganda technique of {readable}."

# 验证所有技术都有定义
missing = [t for t in TECHNIQUES if t not in TECHNIQUE_DEFINITIONS]
if missing:
    print(f"\n⚠ 缺少 Design B 定义：{missing}")
else:
    print("\n✓ 所有技术均有 Design B 定义")

print("\nDesign A 示例:", get_hypothesis_A("Loaded_Language"))
print("Design B 示例:", get_hypothesis_B("Loaded_Language"))

## Cell 3: 数据加载函数

In [ ]:
def load_gold_spans(spans_file: Path) -> Dict[str, List[Dict]]:
    """格式: article_id \t technique \t start \t end"""
    gold = defaultdict(list)
    with open(spans_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) < 4:
                continue
            article_id = parts[0].strip()
            technique  = parts[1].strip()
            start      = int(parts[2])
            end        = int(parts[3])
            gold[article_id].append({"start": start, "end": end, "technique": technique})
    print(f"  {len(gold)} 篇文章，{sum(len(v) for v in gold.values())} spans")
    return dict(gold)


def load_articles(article_dir: Path, article_ids: List[str]) -> Dict[str, str]:
    """加载文章文本，支持 article{id}.txt 和 {id}.txt 两种格式"""

    print(list(article_dir.iterdir())[:5]) # 在 load_articles 函数开头临时加这行，看第一次输出

    articles = {}
    not_found = []
    for aid in article_ids:
        # 尝试带前缀和不带前缀两种文件名
        fname1 = f"{aid}.txt" if aid.startswith("article") else f"article{aid}.txt"
        fname2 = f"{aid}.txt"
        for fname in [fname1, fname2]:
            fpath = article_dir / fname
            if fpath.exists():
                with open(fpath, "r", encoding="utf-8") as f:
                    articles[aid] = f.read()
                break
        else:
            not_found.append(aid)
    if not_found:
        print(f"  ⚠ 未找到 {len(not_found)} 篇（前5）: {not_found[:5]}")
    print(f"  成功加载 {len(articles)} 篇文章")
    return articles


def split_into_paragraphs(text: str, min_len: int = MIN_PARA_LEN) -> List[Dict]:
    """
    先尝试双换行切分；若结果段落数 < 3 且文章足够长，
    回退到单换行切分（适配 EN 文章格式）。
    """
    def _split(text, pattern):
        paragraphs = []
        raw_splits = re.split(pattern, text)
        cursor = 0
        para_idx = 0
        for raw in raw_splits:
            start = text.find(raw, cursor)
            if start == -1:
                cursor += len(raw)
                continue
            end = start + len(raw)
            stripped = raw.strip()
            if len(stripped) >= min_len:
                paragraphs.append({
                    "text": stripped,
                    "start": start,
                    "end": end,
                    "para_idx": para_idx,
                })
                para_idx += 1
            cursor = end
        return paragraphs

    # 先试双换行
    paras = _split(text, r'\n{2,}')

    # 段落太少（< 3）且文章较长 → 回退到单换行
    if len(paras) < 3 and len(text) > 500:
        paras = _split(text, r'\n')

    return paras

def derive_paragraph_gold_labels(
    paragraphs: List[Dict],
    gold_spans: List[Dict],
    techniques: List[str],
) -> List[List[int]]:
    """将字符级 spans 映射到段落级多标签（任意重叠即为正例）"""
    tech_to_idx = {t: i for i, t in enumerate(techniques)}
    para_labels = []
    for para in paragraphs:
        label = [0] * len(techniques)
        p_start, p_end = para["start"], para["end"]
        for span in gold_spans:
            tech = span["technique"]
            if tech not in tech_to_idx:
                continue
            # 任意字符重叠 => 该技术在该段落中存在
            if span["start"] < p_end and span["end"] > p_start:
                label[tech_to_idx[tech]] = 1
        para_labels.append(label)
    return para_labels


print("✓ Data loading functions defined")

In [ ]:
def load_all_articles(article_dir: Path) -> Dict[str, str]:
    """扫描目录下所有 article*.txt，包括 gold=0 的文章"""
    articles = {}
    for fpath in sorted(article_dir.glob("article*.txt")):
        aid = fpath.stem  # "article813452859"
        with open(fpath, "r", encoding="utf-8") as f:
            articles[aid] = f.read()
    print(f"  扫描到 {len(articles)} 篇文章（含 gold=0）")
    return articles

## Cell 4: 加载所有语言数据

In [ ]:
DATA = {}

for lang in ["en", "pl", "ru"]:
    print(f"\n{'='*50} {lang.upper()} {'='*50}")

    gold_spans = load_gold_spans(GOLD_SPAN_PATHS[lang])
    # article_ids = list(gold_spans.keys())

    # articles = load_articles(ARTICLE_DIRS[lang], article_ids)
    articles = load_all_articles(ARTICLE_DIRS[lang])

    paragraphs_all  = {}
    para_gold_all   = {}
    total_paras     = 0
    positive_paras  = 0

    for aid, text in articles.items():
        paras  = split_into_paragraphs(text)
        labels = derive_paragraph_gold_labels(paras, gold_spans.get(aid, []), TECHNIQUES)
        paragraphs_all[aid] = paras
        para_gold_all[aid]  = labels
        total_paras    += len(paras)
        positive_paras += sum(1 for lv in labels if any(lv))

    print(f"  段落总数: {total_paras}，正例段落: {positive_paras} "
          f"({positive_paras/total_paras*100:.1f}%)")

    DATA[lang] = {
        "gold_spans":       gold_spans,
        "articles":         articles,
        "paragraphs":       paragraphs_all,
        "para_gold_labels": para_gold_all,
    }

print("\n✓ 数据加载完成")

## Cell 5: 加载 NLI 模型

In [ ]:
print(f"加载 NLI 模型: {NLI_MODEL}")
print("（首次运行从 HuggingFace 下载约 800MB）")

device = 0 if torch.cuda.is_available() else -1

nli_pipe = pipeline(
    task="zero-shot-classification",
    model=NLI_MODEL,
    device=device,
)

print(f"✓ 模型加载完成（{'GPU' if device == 0 else 'CPU'}）")

# 快速验证
test_result = nli_pipe(
    "The politicians are all corrupt and must be removed!",
    candidate_labels=[get_hypothesis_B("Loaded_Language")],
    hypothesis_template="{}",
    multi_label=True,
)
print("\n验证（Loaded Language 的 entailment 分数应较高）:")
print(f"  {test_result['scores'][0]:.4f}")

## Cell 6: NLI 推理函数

In [ ]:
def run_nli_on_paragraph(text: str, hypotheses: List[str]) -> np.ndarray:
    """
    对单个段落运行 NLI，返回 shape (23,) 的 entailment 概率。
    一次性传入所有 23 个 hypotheses，multi_label=True 保证独立打分。
    """
    result = nli_pipe(
        text,
        candidate_labels=hypotheses,
        hypothesis_template="{}",
        multi_label=True,
    )
    label_to_score = dict(zip(result["labels"], result["scores"]))
    return np.array([label_to_score[h] for h in hypotheses])


def run_nli_on_article(paragraphs: List[Dict], hypotheses: List[str]) -> np.ndarray:
    """对整篇文章的所有段落运行 NLI，返回 shape (num_paras, 23)。"""
    return np.array([run_nli_on_paragraph(p["text"], hypotheses) for p in paragraphs])


print("✓ NLI inference functions defined")

# 快速测试
test_hyps = [get_hypothesis_A(t) for t in TECHNIQUES]
test_scores = run_nli_on_paragraph(
    "Millions of people agree that this is the only solution.",
    test_hyps,
)
top3 = np.argsort(test_scores)[::-1][:3]
print("测试段落 top-3 (Design A):")
for idx in top3:
    print(f"  {TECHNIQUES[idx]}: {test_scores[idx]:.4f}")

## Cell 7: 评估函数（Macro/Micro F1 + 阈值搜索）

In [ ]:
def compute_f1(prob_matrix: np.ndarray, gold_matrix: np.ndarray, threshold: float) -> Dict:
    pred = (prob_matrix >= threshold).astype(int)
    return {
        "threshold":  threshold,
        "macro_f1":   round(f1_score(gold_matrix, pred, average="macro",  zero_division=0) * 100, 2),
        "micro_f1":   round(f1_score(gold_matrix, pred, average="micro",  zero_division=0) * 100, 2),
        "macro_p":    round(precision_score(gold_matrix, pred, average="macro", zero_division=0) * 100, 2),
        "macro_r":    round(recall_score(gold_matrix, pred, average="macro",    zero_division=0) * 100, 2),
        "total_pred": int(pred.sum()),
        "total_gold": int(gold_matrix.sum()),
    }


def threshold_search(
    prob_matrix: np.ndarray,
    gold_matrix: np.ndarray,
    grid: List[float] = THRESHOLD_GRID,
) -> Tuple[Dict, pd.DataFrame]:
    results = [compute_f1(prob_matrix, gold_matrix, thr) for thr in grid]
    df = pd.DataFrame(results)
    best = results[df["macro_f1"].idxmax()]
    print(df.to_string(index=False))
    print(f"\n★ 最优: threshold={best['threshold']}  Macro F1={best['macro_f1']}%")
    return best, df


def per_technique_f1(prob_matrix: np.ndarray, gold_matrix: np.ndarray, threshold: float) -> pd.DataFrame:
    pred = (prob_matrix >= threshold).astype(int)
    rows = []
    for i, tech in enumerate(TECHNIQUES):
        rows.append({
            "technique":   tech,
            "F1":          round(f1_score(gold_matrix[:, i], pred[:, i], zero_division=0) * 100, 2),
            "P":           round(precision_score(gold_matrix[:, i], pred[:, i], zero_division=0) * 100, 2),
            "R":           round(recall_score(gold_matrix[:, i], pred[:, i], zero_division=0) * 100, 2),
            "gold_count":  int(gold_matrix[:, i].sum()),
            "pred_count":  int(pred[:, i].sum()),
        })
    return pd.DataFrame(rows).sort_values("F1", ascending=False)


print("✓ Evaluation functions defined")

## Cell 8: 主实验函数（含缓存机制）

In [ ]:
def run_nli_experiment(lang: str, design: str, data: Dict) -> Dict:
    """
    完整实验：NLI 推理 → 阈值搜索 → per-technique F1 → 保存结果。
    概率矩阵缓存为 .npy，重跑时直接加载。
    """
    print(f"\n{'='*60}")
    print(f"NLI 实验: {lang.upper()} - Design {design}")
    print(f"{'='*60}")

    hypotheses = [get_hypothesis_A(t) for t in TECHNIQUES] if design == "A" \
                 else [get_hypothesis_B(t) for t in TECHNIQUES]

    paragraphs_all  = data["paragraphs"]
    para_gold_all   = data["para_gold_labels"]

    # ---- 缓存路径 ----
    prob_cache = OUTPUT_DIR / f"probs_{lang}_design{design}.npy"
    gold_cache = OUTPUT_DIR / f"gold_{lang}.npy"

    if prob_cache.exists() and gold_cache.exists():
        print(f"  [缓存命中] 加载 {prob_cache.name}")
        prob_matrix = np.load(prob_cache)
        gold_matrix = np.load(gold_cache)
    else:
        all_probs, all_gold = [], []
        article_ids = list(paragraphs_all.keys())
        t0 = time.time()

        for i, aid in enumerate(article_ids):
            paras  = paragraphs_all[aid]
            labels = para_gold_all[aid]
            if not paras:
                continue
            probs = run_nli_on_article(paras, hypotheses)
            all_probs.append(probs)
            all_gold.append(np.array(labels))

            elapsed = time.time() - t0
            eta = elapsed / (i + 1) * (len(article_ids) - i - 1)
            print(f"  [{i+1}/{len(article_ids)}] {aid}: {len(paras)} paras | "
                  f"elapsed={elapsed:.0f}s ETA={eta:.0f}s")

        prob_matrix = np.vstack(all_probs)
        gold_matrix = np.vstack(all_gold)
        np.save(prob_cache, prob_matrix)
        np.save(gold_cache, gold_matrix)
        print(f"  [保存] {prob_cache.name}")

    print(f"  prob shape: {prob_matrix.shape}, gold shape: {gold_matrix.shape}")
    print(f"  正例段落: {(gold_matrix.sum(axis=1)>0).sum()}/{len(gold_matrix)}")

    # ---- 阈值搜索 ----
    print(f"\n--- 阈值搜索 ---")
    best, thr_df = threshold_search(prob_matrix, gold_matrix)

    # ---- Per-technique ----
    print(f"\n--- Per-Technique F1 (threshold={best['threshold']}) ---")
    tech_df = per_technique_f1(prob_matrix, gold_matrix, best["threshold"])
    print(tech_df.to_string(index=False))

    # ---- 保存 ----
    prefix = OUTPUT_DIR / f"nli_{lang}_design{design}"
    thr_df.to_csv(f"{prefix}_threshold.tsv", sep="\t", index=False)
    tech_df.to_csv(f"{prefix}_per_technique.tsv", sep="\t", index=False)
    print(f"  [保存] {prefix}_*.tsv")

    return {"lang": lang, "design": design, **best}


print("✓ Main experiment function defined")

In [ ]:
# ============================================================
# 补全 gold=0 文章的推理（只跑缺失的几篇，不重跑全部）
# ============================================================
import glob

for lang in ["en", "pl"]:  # ru 没有缺失，跳过
    print(f"\n{'='*50} 补全 {lang.upper()} gold=0 文章 {'='*50}")
    
    # 加载已有缓存
    prob_cache = OUTPUT_DIR / f"probs_{lang}_designA.npy"
    gold_cache = OUTPUT_DIR / f"gold_{lang}.npy"
    
    existing_probs = np.load(prob_cache)  # (已有段落数, 23)
    existing_gold  = np.load(gold_cache)
    
    # 找出已经推理过的文章（通过段落数反推）
    # 更直接的方法：记录哪些文章已处理
    paragraphs_all = DATA[lang]["paragraphs"]
    gold_spans     = DATA[lang]["gold_spans"]
    
    # 找出缺失的 gold=0 文章（在目录里有但不在 gold_spans 里）
    all_aids  = set(paragraphs_all.keys())
    gold_aids = set(gold_spans.keys())
    missing_aids = all_aids - gold_aids
    
    print(f"  gold=0 缺失文章数: {len(missing_aids)}: {sorted(missing_aids)}")
    
    if not missing_aids:
        print("  无缺失，跳过")
        continue
    
    # 对缺失文章跑 NLI
    hypotheses = [get_hypothesis_A(t) for t in TECHNIQUES]
    new_probs, new_gold = [], []
    
    for aid in sorted(missing_aids):
        paras = paragraphs_all[aid]
        if not paras:
            continue
        probs = run_nli_on_article(paras, hypotheses)
        new_probs.append(probs)
        # gold=0 文章标签全零
        new_gold.append(np.zeros((len(paras), len(TECHNIQUES)), dtype=int))
        print(f"  {aid}: {len(paras)} paras 推理完成")
    
    # 合并并保存
    combined_probs = np.vstack([existing_probs] + new_probs)
    combined_gold  = np.vstack([existing_gold]  + new_gold)
    
    np.save(prob_cache, combined_probs)
    np.save(gold_cache, combined_gold)
    print(f"  更新后 prob shape: {combined_probs.shape}")
    print(f"  更新后 gold shape: {combined_gold.shape}")
    print(f"  正例段落: {(combined_gold.sum(axis=1)>0).sum()}/{len(combined_gold)}")

In [ ]:
# ============================================================
# 补全 Design B 的 prob 缓存（EN 3篇 + PL 4篇）
# ============================================================
hypotheses_B = [get_hypothesis_B(t) for t in TECHNIQUES]

# ---- EN ----
lang = "en"
prob_cache_B = OUTPUT_DIR / f"probs_{lang}_designB.npy"
existing_probs_B = np.load(prob_cache_B)
print(f"EN Design B 旧 prob shape: {existing_probs_B.shape}")

missing_aids_en = ['article822295249', 'article828866387', 'article832269185']
new_probs_B = []
for aid in missing_aids_en:
    paras = DATA[lang]["paragraphs"][aid]
    probs = run_nli_on_article(paras, hypotheses_B)
    new_probs_B.append(probs)
    print(f"  {aid}: {len(paras)} paras 完成")

combined = np.vstack([existing_probs_B] + new_probs_B)
np.save(prob_cache_B, combined)
print(f"EN Design B 更新后 prob shape: {combined.shape}")

# ---- PL ----
lang = "pl"
prob_cache_B = OUTPUT_DIR / f"probs_{lang}_designB.npy"
existing_probs_B = np.load(prob_cache_B)
print(f"\nPL Design B 旧 prob shape: {existing_probs_B.shape}")

missing_aids_pl = ['article25121', 'article25154', 'article25162', 'article25181']
new_probs_B = []
for aid in missing_aids_pl:
    paras = DATA[lang]["paragraphs"][aid]
    probs = run_nli_on_article(paras, hypotheses_B)
    new_probs_B.append(probs)
    print(f"  {aid}: {len(paras)} paras 完成")

combined = np.vstack([existing_probs_B] + new_probs_B)
np.save(prob_cache_B, combined)
print(f"PL Design B 更新后 prob shape: {combined.shape}")

print("\n✓ 全部补全完成，可以重跑 run_nli_experiment")

## Cell 9: 运行 EN 实验（Design A 先跑验证 pipeline，再跑 Design B）

预计耗时（A40 GPU）：
- EN × 1 Design ≈ 10–20 分钟（~800 段落 × 23 hypotheses）
- 概率矩阵缓存后，重跑阈值搜索 < 1 秒

In [ ]:
summary_en_A = run_nli_experiment("en", "A", DATA["en"])

In [ ]:
summary_en_B = run_nli_experiment("en", "B", DATA["en"])

In [ ]:
# 比较两种 Design
print("=== EN: Design A vs Design B ===")
comp = pd.DataFrame([summary_en_A, summary_en_B])[["design", "threshold", "macro_f1", "micro_f1", "macro_p", "macro_r"]]
print(comp.to_string(index=False))

best_design = "A" if summary_en_A["macro_f1"] >= summary_en_B["macro_f1"] else "B"
print(f"\n★ 最优 Design: {best_design} → 用于 PL & RU")

## Cell 10: 运行 PL & RU

In [ ]:
summary_pl = run_nli_experiment("pl", best_design, DATA["pl"])

In [ ]:
summary_ru = run_nli_experiment("ru", best_design, DATA["ru"])

## Cell 11: 最终汇总表（论文用）

In [ ]:
all_summaries = [summary_en_A, summary_en_B, summary_pl, summary_ru]
results_df = pd.DataFrame(all_summaries)[
    ["lang", "design", "threshold", "macro_f1", "micro_f1", "macro_p", "macro_r"]
]

print("\n=== NLI Zero-Shot Baseline - 最终结果 ===")
print(results_df.to_string(index=False))

summary_path = OUTPUT_DIR / "nli_baseline_summary.tsv"
results_df.to_csv(summary_path, sep="\t", index=False)
print(f"\n✓ 已保存: {summary_path}")

# 与现有方法对比（EN Macro F1）
# 注意：以下数字来自 各个方法checkthat2024的结果，如有差异以该文件为准
print("\n=== 与现有方法对比（EN Macro F1 段落级）===")
comparison = pd.DataFrame([
    {"Method": "NLI-ZS (Design A)", "Type": "Zero-shot",         "EN": f"{summary_en_A['macro_f1']}%", "PL": "—",       "RU": "—"},
    {"Method": "NLI-ZS (Design B)", "Type": "Zero-shot",         "EN": f"{summary_en_B['macro_f1']}%", "PL": "—",       "RU": "—"},
    {"Method": "Sup-FT",            "Type": "Supervised",         "EN": "41.76%",                       "PL": "52.20%",  "RU": "39.10%"},
    {"Method": "Prompt-A",          "Type": "Few-shot",           "EN": "38.47%",                       "PL": "—",       "RU": "—"},
    {"Method": "Iter-Ens",          "Type": "Few-shot+Ensemble",  "EN": "44.88%",                       "PL": "—",       "RU": "—"},
])
print(comparison.to_string(index=False))
print("(PL/RU 的 Sup-FT 数字仅供参考，请以 各个方法checkthat2024的结果 文件为准)")

## Cell 12: 调试 - 检查段落切分与金标准对齐

In [ ]:
# 检查某篇文章的段落切分是否正确
DEBUG_LANG = "en"
DEBUG_IDX  = 0  # 第几篇文章

aids  = list(DATA[DEBUG_LANG]["paragraphs"].keys())
aid   = aids[DEBUG_IDX]
paras = DATA[DEBUG_LANG]["paragraphs"][aid]
labels= DATA[DEBUG_LANG]["para_gold_labels"][aid]
gold  = DATA[DEBUG_LANG]["gold_spans"].get(aid, [])

print(f"文章: {aid}  |  段落数: {len(paras)}  |  spans: {len(gold)}")
for i, (para, label) in enumerate(zip(paras, labels)):
    has_tech = [TECHNIQUES[j] for j, v in enumerate(label) if v]
    print(f"\n[P{i}] char {para['start']}-{para['end']}")
    print(f"  文本: {para['text'][:100]}...")
    print(f"  技术: {has_tech if has_tech else '(无)'}")

## Cell 13: 可选 - 对所有语言跑两种 Design（全矩阵对比）

In [ ]:
# 默认注释掉。如需全对比，取消注释运行

other_design = "B" if best_design == "A" else "A"
summary_pl_other = run_nli_experiment("pl", other_design, DATA["pl"])
summary_ru_other = run_nli_experiment("ru", other_design, DATA["ru"])

print("（可选 cell，默认跳过）")